# Steam Crawler Progress Monitor

This notebook is read-only. It inspects the staged CSV outputs and logs so you can check crawl progress without running the pipeline cells.

In [1]:
from __future__ import annotations

import csv
import gzip
import importlib
import sys
from collections import Counter
from pathlib import Path


ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import steam_crawler.config as steam_config

steam_config = importlib.reload(steam_config)

steam_config.load_project_env(ROOT_DIR, override=True)

DATA_DIR = steam_config.resolve_data_dir(ROOT_DIR)
LOG_DIR = ROOT_DIR / "logs"

STAGE_PATHS = {
    "stage_01": DATA_DIR / "stage_01_apps_catalog.csv",
    "stage_02": DATA_DIR / "stage_02_app_details.csv.gz",
    "stage_03": DATA_DIR / "stage_03_apps_with_metadata.csv.gz",
    "stage_04": DATA_DIR / "stage_04_selected_games.csv",
    "stage_05": DATA_DIR / "stage_05_reviews_dataset.csv.gz",
    "stage_05_progress": DATA_DIR / "stage_05_progress.csv",
    "errors": LOG_DIR / "errors.csv",
    "run_log": LOG_DIR / "run.log",
}

APPID_TO_INSPECT = None

print(f"ROOT_DIR: {ROOT_DIR}")
for name, path in STAGE_PATHS.items():
    print(f"{name:>16}: {path}")

ROOT_DIR: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler
        stage_01: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_01_apps_catalog.csv
        stage_02: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_02_app_details.csv.gz
        stage_03: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_03_apps_with_metadata.csv.gz
        stage_04: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04_selected_games.csv
        stage_05: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05_reviews_dataset.csv.gz
stage_05_progress: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05_progress.csv
          errors: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/logs/errors.csv
         run_log: /Users/gitaa

In [2]:
CSV_FIELD_SIZE_LIMIT_READY = False


def configure_csv_field_size_limit() -> None:
    global CSV_FIELD_SIZE_LIMIT_READY
    if CSV_FIELD_SIZE_LIMIT_READY:
        return
    limit = sys.maxsize
    while True:
        try:
            csv.field_size_limit(limit)
            CSV_FIELD_SIZE_LIMIT_READY = True
            return
        except OverflowError:
            limit //= 10


def open_text(path: Path):
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8", newline="")
    return path.open("rt", encoding="utf-8", newline="")


def iter_csv_rows(path: Path):
    configure_csv_field_size_limit()
    with open_text(path) as handle:
        yield from csv.DictReader(handle)


def count_csv_rows(path: Path) -> int:
    if not path.exists():
        return 0
    configure_csv_field_size_limit()
    with open_text(path) as handle:
        reader = csv.reader(handle)
        next(reader, None)
        return sum(1 for _ in reader)


def tail_rows(path: Path, limit: int = 10) -> list[dict[str, str]]:
    if not path.exists():
        return []
    rows = list(iter_csv_rows(path))
    return rows[-limit:]


def format_ratio(numerator: int, denominator: int) -> str:
    if denominator <= 0:
        return "n/a"
    return f"{(numerator / denominator) * 100:.2f}%"


def print_section(title: str) -> None:
    print("\n" + title)
    print("-" * len(title))


def summarize_stage_outputs() -> dict[str, int]:
    counts = {
        "stage_01": count_csv_rows(STAGE_PATHS["stage_01"]),
        "stage_02": count_csv_rows(STAGE_PATHS["stage_02"]),
        "stage_03": count_csv_rows(STAGE_PATHS["stage_03"]),
        "stage_04": count_csv_rows(STAGE_PATHS["stage_04"]),
        "stage_05": count_csv_rows(STAGE_PATHS["stage_05"]),
        "stage_05_progress": count_csv_rows(STAGE_PATHS["stage_05_progress"]),
        "errors": count_csv_rows(STAGE_PATHS["errors"]),
    }
    return counts


def eligible_summary(path: Path) -> tuple[int, int]:
    if not path.exists():
        return 0, 0
    eligible = 0
    total = 0
    for row in iter_csv_rows(path):
        total += 1
        if str(row.get("eligible_for_sampling", "")).lower() == "true":
            eligible += 1
    return eligible, total


def stage_05_status_counts(path: Path) -> Counter:
    counter: Counter = Counter()
    if not path.exists():
        return counter
    latest_by_appid: dict[str, dict[str, str]] = {}
    for row in iter_csv_rows(path):
        latest_by_appid[row.get("appid", "")] = row
    for row in latest_by_appid.values():
        counter[row.get("status", "unknown") or "unknown"] += 1
    return counter


def review_counts_by_app(path: Path, top_n: int = 10) -> list[tuple[str, int]]:
    counter: Counter = Counter()
    if not path.exists():
        return []
    for row in iter_csv_rows(path):
        counter[row.get("appid", "")] += 1
    return counter.most_common(top_n)


def recent_error_counts(path: Path) -> Counter:
    counter: Counter = Counter()
    if not path.exists():
        return counter
    for row in iter_csv_rows(path):
        stage = row.get("stage", "unknown") or "unknown"
        status = row.get("status_code", "") or "exc"
        counter[f"{stage}:{status}"] += 1
    return counter


In [3]:
counts = summarize_stage_outputs()
eligible_count, eligible_total = eligible_summary(STAGE_PATHS["stage_03"])
status_counts = stage_05_status_counts(STAGE_PATHS["stage_05_progress"])

print_section("Stage Output Counts")
for name in ["stage_01", "stage_02", "stage_03", "stage_04", "stage_05", "stage_05_progress", "errors"]:
    print(f"{name:>16}: {counts[name]:>12,}")

print_section("Coverage")
print(f"Stage 2 coverage vs Stage 1: {counts['stage_02']:,} / {counts['stage_01']:,} ({format_ratio(counts['stage_02'], counts['stage_01'])})")
print(f"Stage 3 coverage vs Stage 1: {counts['stage_03']:,} / {counts['stage_01']:,} ({format_ratio(counts['stage_03'], counts['stage_01'])})")
print(f"Eligible-for-sampling rows in Stage 3: {eligible_count:,} / {eligible_total:,} ({format_ratio(eligible_count, eligible_total)})")
print(f"Selected games in Stage 4: {counts['stage_04']:,}")
print(f"Review rows in Stage 5: {counts['stage_05']:,}")

print_section("Stage 5 Status Counts")
if status_counts:
    for status, count in sorted(status_counts.items()):
        print(f"{status:>16}: {count:>12,}")
else:
    print("stage_05_progress.csv not present yet")


Stage Output Counts
-------------------
        stage_01:      162,305
        stage_02:      162,304
        stage_03:      162,305
        stage_04:        9,598
        stage_05:            0
stage_05_progress:            0
          errors:        9,074

Coverage
--------
Stage 2 coverage vs Stage 1: 162,304 / 162,305 (100.00%)
Stage 3 coverage vs Stage 1: 162,305 / 162,305 (100.00%)
Eligible-for-sampling rows in Stage 3: 9,598 / 162,305 (5.91%)
Selected games in Stage 4: 9,598
Review rows in Stage 5: 0

Stage 5 Status Counts
---------------------
stage_05_progress.csv not present yet


In [4]:
print_section("Top Review Counts by App")
top_review_counts = review_counts_by_app(STAGE_PATHS["stage_05"], top_n=15)
if not top_review_counts:
    print("stage_05_reviews_dataset.csv.gz not present yet")
else:
    for appid, count in top_review_counts:
        print(f"appid={appid:>10} | reviews={count:>6,}")


Top Review Counts by App
------------------------
stage_05_reviews_dataset.csv.gz not present yet


In [5]:
print_section("Recent Error Buckets")
error_counts = recent_error_counts(STAGE_PATHS["errors"])
if not error_counts:
    print("logs/errors.csv not present yet")
else:
    for label, count in error_counts.most_common(12):
        print(f"{label:>20}: {count:>8,}")

print_section("Last 10 Error Rows")
for row in tail_rows(STAGE_PATHS["errors"], limit=10):
    stage = row.get("stage", "")
    appid = row.get("appid", "")
    status = row.get("status_code", "") or "exc"
    retry_after = row.get("retry_after_seconds", "") or "-"
    exc_type = row.get("exception_type", "")
    exc_message = row.get("exception_message", "")
    print(f"[{stage}] appid={appid or '-'} status={status} retry_after={retry_after} {exc_type}: {exc_message}")


Recent Error Buckets
--------------------
        stage_02:429:    9,074

Last 10 Error Rows
------------------
[stage_02] appid=1239740 status=429 retry_after=301.0 HTTPError: 429 Client Error: Too Many Requests for url: https://store.steampowered.com/api/appdetails?appids=1239740&cc=us&l=english&filters=basic%2Ccategories%2Crecommendations
[stage_02] appid=1246120 status=429 retry_after=301.0 HTTPError: 429 Client Error: Too Many Requests for url: https://store.steampowered.com/api/appdetails?appids=1246120&cc=us&l=english&filters=basic%2Ccategories%2Crecommendations
[stage_02] appid=1251670 status=429 retry_after=301.0 HTTPError: 429 Client Error: Too Many Requests for url: https://store.steampowered.com/api/appdetails?appids=1251670&cc=us&l=english&filters=basic%2Ccategories%2Crecommendations
[stage_02] appid=1257000 status=429 retry_after=301.0 HTTPError: 429 Client Error: Too Many Requests for url: https://store.steampowered.com/api/appdetails?appids=1257000&cc=us&l=english&filt

In [6]:
print_section("Optional App Inspection")
if APPID_TO_INSPECT is None:
    print("Set APPID_TO_INSPECT in the first code cell to inspect one app across Stage 2/4/5 outputs.")
else:
    appid_text = str(APPID_TO_INSPECT)
    stage_02_rows = [row for row in iter_csv_rows(STAGE_PATHS['stage_02']) if row.get('appid') == appid_text] if STAGE_PATHS['stage_02'].exists() else []
    stage_04_rows = [row for row in iter_csv_rows(STAGE_PATHS['stage_04']) if row.get('appid') == appid_text] if STAGE_PATHS['stage_04'].exists() else []
    stage_05_progress_rows = [row for row in iter_csv_rows(STAGE_PATHS['stage_05_progress']) if row.get('appid') == appid_text] if STAGE_PATHS['stage_05_progress'].exists() else []
    stage_05_review_count = sum(1 for row in iter_csv_rows(STAGE_PATHS['stage_05']) if row.get('appid') == appid_text) if STAGE_PATHS['stage_05'].exists() else 0

    print(f"appid={appid_text}")
    print(f"  stage_02 rows: {len(stage_02_rows)}")
    if stage_02_rows:
        row = stage_02_rows[-1]
        print(f"  stage_02 success: {row.get('success')}")
        print(f"  stage_02 type: {row.get('type')}")
        print(f"  recommendations_total: {row.get('recommendations_total')}")
    print(f"  selected in stage_04: {'yes' if stage_04_rows else 'no'}")
    print(f"  review rows in stage_05: {stage_05_review_count}")
    if stage_05_progress_rows:
        latest = stage_05_progress_rows[-1]
        print(f"  latest stage_05 status: {latest.get('status')}")
        print(f"  latest phase: {latest.get('phase')}")
        print(f"  total_unique: {latest.get('total_unique')}")
        print(f"  error: {latest.get('error')}")


Optional App Inspection
-----------------------
Set APPID_TO_INSPECT in the first code cell to inspect one app across Stage 2/4/5 outputs.
